# 19: Text Generation - Creating New Text

## Generative Models

So far we've done **classification** (predict labels). Now: **generation** (create new sequences)!

**Character-level generation**:
- Input: sequence of characters
- Output: probability distribution over next character
- Sample from distribution to generate text

### The Web Dev Analogy

Text generation is like **autocomplete**:
- Model learns patterns from training data
- Given context, predicts what comes next
- Sample predictions to generate novel text
- Temperature controls creativity vs consistency

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to generate text! ✍️")

## 1. Preparing Text Data

In [ ]:
# Sample text (in practice, use much more!)
text = """Hello world! Machine learning is amazing. Neural networks can learn patterns from data. 
Deep learning uses multiple layers. Transformers are powerful models. 
Text generation creates new sequences. Language models predict the next word."""

print(f"Text length: {len(text)} characters")
print(f"\nFirst 100 characters:")
print(text[:100])

# Create character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f"\nVocabulary ({vocab_size} unique characters):")
print(''.join(chars))

# Create mappings
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"\nExample mappings:")
for ch in ['H', 'e', 'l', 'o', ' ', '.']:
    print(f"  '{ch}' → {char_to_idx[ch]}")

In [ ]:
# Convert text to indices
data = torch.tensor([char_to_idx[ch] for ch in text], dtype=torch.long)

print(f"Text as indices: {data[:20].tolist()}...")
print(f"Shape: {data.shape}")

# Verify we can decode
decoded = ''.join([idx_to_char[idx.item()] for idx in data[:20]])
print(f"\nDecoded back: '{decoded}'")

## 2. Creating Training Sequences

In [ ]:
# Create input-output pairs
seq_length = 25  # Use 25 characters to predict the 26th

def create_sequences(data, seq_length):
    sequences = []
    targets = []
    
    for i in range(len(data) - seq_length):
        seq = data[i:i+seq_length]
        target = data[i+seq_length]
        sequences.append(seq)
        targets.append(target)
    
    return torch.stack(sequences), torch.stack(targets)

X, y = create_sequences(data, seq_length)

print(f"Input sequences: {X.shape}")
print(f"Target characters: {y.shape}")

# Show example
idx = 0
input_text = ''.join([idx_to_char[i.item()] for i in X[idx]])
target_char = idx_to_char[y[idx].item()]

print(f"\nExample:")
print(f"  Input:  '{input_text}'")
print(f"  Target: '{target_char}'")
print(f"\n💡 Model learns: given sequence, predict next character")

## 3. Character-Level LSTM

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Embed characters
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # LSTM layers
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
        
        # Output layer (vocab_size classes)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden=None):
        # x: (batch, seq_len)
        embedded = self.embedding(x)  # (batch, seq_len, embedding_dim)
        
        # LSTM
        if hidden is None:
            output, hidden = self.lstm(embedded)
        else:
            output, hidden = self.lstm(embedded, hidden)
        
        # output: (batch, seq_len, hidden_dim)
        
        # Reshape for linear layer
        output = output.reshape(-1, self.hidden_dim)
        
        # Predict next character
        logits = self.fc(output)  # (batch * seq_len, vocab_size)
        
        return logits, hidden

# Create model
model = CharLSTM(
    vocab_size=vocab_size,
    embedding_dim=32,
    hidden_dim=128,
    num_layers=2
)

print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

`★ Insight ─────────────────────────────────────`

**How text generation works:**
1. **Training**: Learn P(next_char | previous_chars)
2. **Generation**: Sample from learned distribution
3. **Autoregressive**: Feed generated char back as input

Model outputs logits over vocabulary → softmax → probability distribution

`─────────────────────────────────────────────────`

## 4. Training the Model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

n_epochs = 100
batch_size = 32
history = []

print("Training character-level LSTM...")
print("=" * 70)

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    n_batches = 0
    
    # Mini-batch training
    for i in range(0, len(X), batch_size):
        batch_X = X[i:i+batch_size]
        batch_y = y[i:i+batch_size]
        
        # Forward
        logits, _ = model(batch_X)
        loss = criterion(logits, batch_y)
        
        # Backward
        loss.backward()
        
        # Clip gradients (helps with exploding gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        
        optimizer.step()
        optimizer.zero_grad()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    history.append(avg_loss)
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}: Loss = {avg_loss:.4f}")

print(f"\n✅ Training complete! Final loss: {history[-1]:.4f}")

In [ ]:
# Visualize loss
plt.figure(figsize=(10, 6))
plt.plot(history, linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Generating Text

In [ ]:
def generate_text(model, start_text, length=100, temperature=1.0):
    """Generate text starting from start_text.
    
    Temperature:
    - < 1: More conservative (peaks around likely chars)
    - = 1: Original distribution
    - > 1: More creative (flattens distribution)
    """
    model.train(False)
    
    # Convert start text to indices
    chars_idx = [char_to_idx[ch] for ch in start_text]
    input_seq = torch.tensor(chars_idx).unsqueeze(0)  # (1, seq_len)
    
    generated = start_text
    hidden = None
    
    with torch.no_grad():
        for _ in range(length):
            # Forward pass
            logits, hidden = model(input_seq, hidden)
            
            # Get last character's logits
            logits = logits[-1, :] / temperature
            
            # Sample from probability distribution
            probs = F.softmax(logits, dim=0)
            next_idx = torch.multinomial(probs, 1).item()
            
            # Append to generated text
            next_char = idx_to_char[next_idx]
            generated += next_char
            
            # Update input (use only last character for next prediction)
            input_seq = torch.tensor([[next_idx]])
    
    return generated

# Generate with different temperatures
start = "Neural"
print(f"Starting text: '{start}'\n")
print("=" * 70)

for temp in [0.5, 1.0, 1.5]:
    text = generate_text(model, start, length=100, temperature=temp)
    print(f"\nTemperature {temp}:")
    print(text)
    print("-" * 70)

## 6. Temperature: Controlling Creativity

In [ ]:
# Visualize temperature effect on probability distribution
# Simulate a probability distribution
logits = torch.tensor([2.0, 1.0, 0.5, 0.2, 0.1])
temperatures = [0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, temp in enumerate(temperatures):
    probs = F.softmax(logits / temp, dim=0).numpy()
    
    axes[idx].bar(range(len(probs)), probs, color='steelblue')
    axes[idx].set_title(f'Temperature = {temp}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Character')
    axes[idx].set_ylabel('Probability')
    axes[idx].set_ylim([0, 1])
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Temperature effects:")
print("  Low (0.5):  Peaked → more conservative, repeats patterns")
print("  Medium (1.0): Original → balanced")
print("  High (2.0):  Flat → more creative, more random")

`★ Insight ─────────────────────────────────────`

**Key concepts in text generation:**
1. **Autoregressive**: Each output becomes next input
2. **Sampling**: Don't just take argmax - sample for variety
3. **Temperature**: Control exploration vs exploitation
4. **Hidden state**: Carries context through generation

Modern models (GPT, etc.) use same principles at word/token level!

`─────────────────────────────────────────────────`

## 7. Evaluation: Perplexity

In [ ]:
# Perplexity = exp(average loss)
# Lower is better (model is less "perplexed" by the data)

def compute_perplexity(model, X, y):
    model.train(False)
    total_loss = 0
    
    with torch.no_grad():
        logits, _ = model(X)
        loss = criterion(logits, y)
        total_loss = loss.item()
    
    perplexity = np.exp(total_loss)
    return perplexity

perplexity = compute_perplexity(model, X, y)
print(f"Perplexity: {perplexity:.2f}")
print(f"\n💡 Lower perplexity = better model")
print(f"   Random guessing: ~{vocab_size:.0f}")
print(f"   Our model: {perplexity:.2f}")

if perplexity < vocab_size:
    print(f"   ✅ Much better than random!")
else:
    print(f"   ⚠️  Need more training or data!")

## 8. Improving Generation Quality

Ways to get better results:
1. **More data**: Larger training corpus
2. **Longer sequences**: Capture longer context
3. **Bigger model**: More layers, larger hidden size
4. **Word-level**: Use words instead of characters
5. **Attention**: Add attention mechanism (Module 7!)
6. **Transformers**: State-of-the-art architecture (Module 8!)

In [ ]:
print("Tips for better text generation:")
print("=" * 70)

print("\n1. Data Quality:")
print("   - Clean, consistent text")
print("   - Sufficient quantity (ideally millions of characters)")
print("   - Relevant to your target domain")

print("\n2. Model Architecture:")
print("   - Start with 2-3 LSTM layers")
print("   - Hidden size: 256-512")
print("   - Add dropout (0.2-0.5) to prevent overfitting")

print("\n3. Training:")
print("   - Use gradient clipping (prevents exploding gradients)")
print("   - Monitor perplexity on validation set")
print("   - Learning rate scheduling")

print("\n4. Generation:")
print("   - Experiment with temperature (0.5-1.5)")
print("   - Try top-k sampling (sample from top k most likely)")
print("   - Use nucleus (top-p) sampling for quality")

print("\n5. Evaluation:")
print("   - Perplexity (quantitative)")
print("   - Human evaluation (qualitative)")
print("   - Check for repetition and coherence")

## 📝 Check Your Understanding

1. What is autoregressive generation?
2. Why do we sample instead of using argmax?
3. What does temperature control?
4. What is perplexity and why is lower better?
5. Why is gradient clipping important for RNNs?

## 🎯 Summary

**Text generation with RNNs**:
- Train to predict next character/word
- Generate by sampling from predicted distribution
- Feed generated output back as input (autoregressive)

**Key techniques**:
- **Temperature sampling**: Control creativity
- **Gradient clipping**: Prevent exploding gradients
- **Hidden state**: Maintain context during generation

**Metrics**:
- **Loss**: Training objective
- **Perplexity**: exp(loss), measures uncertainty
- **Human eval**: Ultimate quality measure

**Limitations of RNNs**:
- Sequential processing (slow)
- Still struggle with very long context
- No explicit mechanism to focus on relevant parts

**Next up**: Attention mechanism - the breakthrough that changed everything! →